# 🎓 GaussianEditor: A Comprehensive Learning Journey

Welcome to the guided tutorial for **GaussianEditor**! This repository enables swift and controllable 3D editing using Gaussian Splatting and 2D Diffusion Models. 

If you are encountering this codebase for the first time, this notebook will break down its architecture into digestible, interconnected lessons. You will learn how we represent a 3D scene, how we render it to 2D, how we edit that 2D image, and finally, how we update the 3D scene to match the edit.

---

## 📑 Table of Contents
1. [High-Level Architecture Overview](#1.-High-Level-Architecture-Overview)
2. [Core Building Blocks in Sequence](#2.-Core-Building-Blocks-in-Sequence)
    * [Block 1: Gaussian Splatting Representation](#Block-1:-Gaussian-Splatting-Representation)
    * [Block 2: The Renderer](#Block-2:-The-Renderer)
    * [Block 3: Diffusion Guidance (The Editor)](#Block-3:-Diffusion-Guidance-(The-Editor))
3. [Data Flow Walkthrough](#3.-Data-Flow-Walkthrough)
4. [Integration Points](#4.-Integration-Points)
5. [Practical Examples](#5.-Practical-Examples)
6. [Common Pitfalls and Design Decisions](#6.-Common-Pitfalls-and-Design-Decisions)

In [ ]:
# Setup: Run this cell first to ensure all paths are correctly configured.
import sys
import os

# Add the repository root to the python path so we can import modules correctly.
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.append(repo_root)

# Standard libraries
import torch
import torch.nn.functional as F
import numpy as np

# Verify GPU availability (Crucial for Gaussian Splatting)
if not torch.cuda.is_available():
    print("⚠️ WARNING: No CUDA GPU detected. Gaussian Splatting requires a CUDA-enabled GPU.")
else:
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    
print("Environment setup complete.")

---
<a id="1.-High-Level-Architecture-Overview"></a>
## 1. High-Level Architecture Overview

Before looking at code, let's understand the system visually and conceptually.

GaussianEditor bridges two distinct fields of AI:
1. **3D Representation:** Representing a 3D scene using millions of tiny blobs (Gaussians).
2. **2D Generative AI:** Using powerful 2D diffusion models (like InstructPix2Pix) to alter images based on text.

### The Flowchart

```text
[ 3D Scene (GaussianModel) ] 
       |
       | (Render from Camera Viewpoint)
       v
[ 2D Rendered Image (pred_rgb) ]
       |
       | (Add noise, apply Text Prompt: "Make it snowy")
       v
[ Diffusion Guidance Model (InstructPix2Pix / ControlNet) ]
       |
       | (Calculates Score Distillation Sampling loss - how to change the image)
       v
[ Gradient Calculation (loss.backward) ]
       |
       | (Updates the properties of the 3D blobs: position, color, opacity)
       v
[ Updated 3D Scene ] ---> (Repeat from a new camera angle)
```

By repeating this process from multiple angles, a 2D edit successfully transfers into a coherent 3D edit!

### How the Repository is Organized
* `gaussiansplatting/`: The core 3D engine. Contains `GaussianModel` (the data structure) and `gaussian_renderer` (the rendering engine).
* `GUI/` & `threestudio/models/guidance/`: The "magical artist". Handles passing rendered images to Diffusion Models.
* `threestudio/systems/`: The conductor. It ties the renderer, the dataset (cameras), and the guidance model together into a training loop.

---
<a id="2.-Core-Building-Blocks-in-Sequence"></a>
## 2. Core Building Blocks in Sequence

Let's break down the fundamental components in isolation.

### <a id="Block-1:-Gaussian-Splatting-Representation"></a> Block 1: Gaussian Splatting Representation

In Gaussian Splatting, a 3D scene isn't made of triangles. It's made of **Gaussians**. Each Gaussian is defined by:
* **Position ($\mu$)**: Its `xyz` coordinates in 3D space.
* **Covariance ($\Sigma$)**: Its 3D shape, controlled by `scaling` and `rotation`.
* **Opacity ($\alpha$)**: How solid it is.
* **Color**: Defined by Spherical Harmonics (`features_dc` and `features_rest`) to allow color to change based on viewing angle.

Let's initialize the actual `GaussianModel` from the repository.

In [ ]:
from gaussiansplatting.scene.gaussian_model import GaussianModel

# We initialize it with the degree of spherical harmonics (3 is standard for high detail)
sh_degree = 3
scene_gaussians = GaussianModel(sh_degree)

# In a real scenario, you would load a point cloud from COLMAP:
# scene_gaussians.load_ply("path/to/colmap/point_cloud.ply")

print(f"Initialized GaussianModel.")
print(f"It holds PyTorch Parameters for:")
print(f"  - _xyz (shape: {scene_gaussians._xyz.shape if scene_gaussians._xyz.numel() > 0 else 'Empty'})")
print(f"  - _opacity (shape: {scene_gaussians._opacity.shape if scene_gaussians._opacity.numel() > 0 else 'Empty'})")
print(f"  - _scaling (shape: {scene_gaussians._scaling.shape if scene_gaussians._scaling.numel() > 0 else 'Empty'})")


### <a id="Block-2:-The-Renderer"></a> Block 2: The Renderer

The renderer projects the 3D Gaussians onto a 2D plane (your screen) based on a `Camera` object.
Because the Gaussians are analytically defined, the math is fast and **differentiable**. If we change a pixel in the rendered image, PyTorch calculates exactly how to adjust the 3D Gaussians to make that change happen!

In [ ]:
from gaussiansplatting.gaussian_renderer import render
from gaussiansplatting.scene.cameras import Camera

# Let's create a dummy camera looking at the origin
dummy_camera = Camera(
    colmap_id=0, R=np.eye(3), T=np.zeros(3),
    FoVx=np.pi/3, FoVy=np.pi/3,
    image=torch.zeros((3, 512, 512)), # Dummy GT image
    gt_alpha_mask=None, image_name="dummy", uid=0
)

# Set background color to black
bg_color = torch.tensor([0.0, 0.0, 0.0], dtype=torch.float32, device="cuda" if torch.cuda.is_available() else "cpu")

# We can't actually render an empty scene without CUDA errors, but this is the API:
print("Rendering API:")
print("render_pkg = render(dummy_camera, scene_gaussians, pipeline_settings, bg_color)")
print("rendered_image = render_pkg['render']  # This returns a [3, H, W] tensor")

# If you had loaded points and had a GPU, this would output the 2D view!


### <a id="Block-3:-Diffusion-Guidance-(The-Editor)"></a> Block 3: Diffusion Guidance (The Editor)

This component takes the rendered image and a text prompt, and uses a pre-trained Diffusion Model to score how well the image matches the prompt.

The repository supports multiple guidance types through `threestudio` and `GUI/` scripts. Here is a conceptual example of how the `GUI/EditGuidance.py` module operates.

In [ ]:
# The guidance models wrap HuggingFace diffusers pipelines.
# For example, stable-diffusion-instructpix2pix-guidance.

print("Guidance Module API:")
print("1. It accepts `pred_rgb` (the rendered image) and `prompt` ('make it snowy').")
print("2. It adds noise to `pred_rgb`.")
print("3. It asks the Diffusion UNet to predict the noise given the text prompt.")
print("4. It calculates Score Distillation Sampling (SDS) loss or Delta Denoising Score (DDS).")
print("5. Returns a dictionary: {'loss_sds': tensor(...)}")

# Example of what the output looks like:
dummy_loss = torch.tensor(0.5, requires_grad=True)
print(f"Guidance Loss Output: {dummy_loss}")


---
<a id="3.-Data-Flow-Walkthrough"></a>
## 3. Data Flow Walkthrough

Let's trace how data moves through the system end-to-end. When you run the training script, `threestudio` executes a loop similar to this:

In [ ]:
# --- The End-to-End Editing Loop (Simplified) ---

# 1. Setup
optimizer = torch.optim.Adam(scene_gaussians.parameters(), lr=0.001)

# We define an Anchor Loss (crucial for GaussianEditor)
# This prevents the scene from collapsing by penalizing Gaussians that stray too far from their original positions.
def anchor_loss(current_xyz, original_xyz):
    return F.mse_loss(current_xyz, original_xyz)

print("Starting Editing Loop...")

# For demonstration, we assume we have a list of camera viewpoints
# camera_batches = [dummy_camera_1, dummy_camera_2, ...]

# Training loop
# for step in range(num_steps):
#     optimizer.zero_grad()
#     
#     # Step A: Render from a random camera viewpoint
#     camera = get_random_camera()
#     render_pkg = render(camera, scene_gaussians, pipe, bg_color)
#     pred_rgb = render_pkg["render"]
#     
#     # Step B: Pass the rendered image to Diffusion Guidance
#     guidance_outputs = editor(pred_rgb, prompt="Make it a winter scene")
#     loss_sds = guidance_outputs["loss_sds"]
#     
#     # Step C: Add Anchor Loss to preserve geometry
#     loss_anchor = anchor_loss(scene_gaussians.get_xyz(), original_gaussians_xyz)
#     total_loss = loss_sds + (lambda_anchor_geo * loss_anchor)
#     
#     # Step D: Backpropagate and update Gaussians
#     total_loss.backward()
#     optimizer.step()
#     
#     # Step E: Densification (Optional)
#     # scene_gaussians.densify_and_prune(...)
    
print("Editing Loop Complete! The 3D scene is now updated.")


---
<a id="4.-Integration-Points"></a>
## 4. Integration Points

The magic that ties everything together is the `threestudio` configuration system.

If you look inside `configs/edit-ctn.yaml`, you will see:
```yaml
system_type: "gsedit-system-edit"
system:
  gs_source: <path_to_ply>
  prompt_processor:
    prompt: 'a bicycle parked next to a bench in a park, all covered with snow, winter'
  loss:
    lambda_anchor_color: 5
    lambda_anchor_geo: 50
```

* `system_type: "gsedit-system-edit"` tells `threestudio` to load the class inside `threestudio/systems/GassuianEditorEdit.py`.
* This class inherits from `BaseSystem` (`threestudio/systems/base.py`) which defines the `training_step` that implements the loop we just walked through.

In [ ]:
import threestudio
from threestudio.systems.base import BaseSystem

# Threestudio uses a registry system. When you define a system class, it registers itself.
# We can dynamically retrieve the class based on the yaml config:

try:
    system_class = threestudio.find("gsedit-system-edit")
    print(f"Successfully located system class: {system_class.__name__}")
except Exception as e:
    print(f"Could not load system class (make sure threestudio dependencies are met): {e}")

print("This system class coordinates the Geometry, Material, Background, Renderer, and Guidance based on your config file.")


---
<a id="5.-Practical-Examples"></a>
## 5. Practical Examples

To run an edit on your machine, you use the `launch.py` script.

### Command Line Editing
```bash
python launch.py \
    --config configs/edit-ctn.yaml \
    --train \
    --gpu 0 \
    system.prompt_processor.prompt="make the grass on fire" \
    data.source=/path/to/colmap/data \
    system.gs_source=/path/to/pretrained/point_cloud.ply
```

### Interactive WebUI
GaussianEditor also provides a WebUI powered by Viser!
```bash
python webui.py
```
This allows you to draw inpainting masks, type prompts, and watch the 3D scene update in real-time in your browser.

---
<a id="6.-Common-Pitfalls-and-Design-Decisions"></a>
## 6. Common Pitfalls and Design Decisions

### 1. Why do we need `lambda_anchor` losses?
If you only tell a diffusion model "Make this image look snowy", it might decide to completely change the geometry of the scene to draw a giant snowman, ignoring the original 3D structure.
* **Design Decision:** The system uses **Anchor Losses** (`lambda_anchor_geo`, `lambda_anchor_color`). It keeps a copy of the original, unedited Gaussians. During training, it heavily penalizes the edited Gaussians if they move too far away from their original positions or change color drastically where they shouldn't.

### 2. The Densification Dilemma
In Gaussian Splatting, "Densification" is the process of splitting one big Gaussian into two smaller ones, or cloning a Gaussian, to add more detail.
* **Pitfall:** If you densify too much while editing, the scene becomes a chaotic mess of tiny blobs.
* **Solution:** Notice in `configs/edit-ctn.yaml`, `densify_until_iter` and `max_densify_percent` are carefully tuned. Often, it's safer to limit densification during editing to preserve the structure.

### 3. "Bad result for Edit" (from the FAQ)
* **Pitfall:** InstructPix2Pix is incredibly powerful but highly sensitive to the exact phrasing of the prompt.
* **Solution:** Always test your prompt on a 2D InstructPix2Pix demo first. If it can't edit a single 2D image correctly, the 3D Gaussian editor will definitely fail!

---
## 🎉 Conclusion
You now understand the anatomy of GaussianEditor! You've learned about the Gaussian representation, the renderer, the guidance models, how `threestudio` orchestrates the training loop, and the crucial design decisions that make 3D editing possible.